# 03 — Analyze failures

Drills into the incorrect/unclear answers of an evaluation run: expected vs actual, a link to the
agent thread, and the **generated queries + raw tool outputs** captured by the SDK
(`evaluation_output_steps`).

In [ ]:
%%configure -f
{
    "defaultLakehouse": {
        "name": "{{DEFAULT_LAKEHOUSE_NAME}}",
        "id": "{{DEFAULT_LAKEHOUSE_ID}}",
        "workspaceId": "{{WORKSPACE_ID}}"
    }
}

In [ ]:
# PARAMETERS -----------------------------------------------------------------
RUN_ID = ""          # evaluation_id; empty = most recent run
SHOW_CORRECT = False  # include correct answers in the detail listing
OUTPUT_TABLE = "evaluation_output"

In [ ]:
from pyspark.sql import functions as F

out = spark.read.table(OUTPUT_TABLE)
if not RUN_ID:
    RUN_ID = (out.orderBy(F.desc("run_timestamp")).select("evaluation_id").first() or [None])[0]
    if not RUN_ID:
        raise ValueError("No evaluation runs found - run 02_run_evaluation first.")
print("Analyzing run:", RUN_ID)

rows = out.filter(F.col("evaluation_id") == RUN_ID)
steps = spark.read.table(OUTPUT_TABLE + "_steps")
detail = rows.join(steps.select("id", "function_queries", "function_outputs",
                                "sql_steps", "dax_steps", "kql_steps"), "id", "left").toPandas()

n = len(detail)
n_true = int((detail["evaluation_judgement"] == True).sum())
n_false = int((detail["evaluation_judgement"] == False).sum())
n_unclear = int(detail["evaluation_judgement"].isna().sum())
print(f"{n} question(s): {n_true} correct, {n_false} incorrect, {n_unclear} unclear "
      f"(accuracy {n_true / n:.0%})" if n else "empty run")

In [ ]:
# Failure detail ---------------------------------------------------------------
import ast

def _fmt_steps(val):
    """steps columns hold stringified lists; keep non-None entries"""
    try:
        items = ast.literal_eval(val) if isinstance(val, str) else (val or [])
        return [x for x in items if x and x != "None"]
    except Exception:
        return [str(val)] if val else []

shown = detail if SHOW_CORRECT else detail[detail["evaluation_judgement"] != True]
if shown.empty:
    print("No failures in this run.")
for _, r in shown.iterrows():
    verdict = {True: "CORRECT", False: "INCORRECT"}.get(r["evaluation_judgement"], "UNCLEAR")
    print("=" * 100)
    print(f"[{verdict}] {r['question']}")
    print(f"  expected: {r['expected_answer']}")
    print(f"  actual:   {str(r['actual_answer'])[:400]}")
    print(f"  latency:  {r['execution_time_sec']}s   thread: {r['thread_url']}")
    for col in ("sql_steps", "dax_steps", "kql_steps", "function_queries"):
        vals = _fmt_steps(r.get(col))
        if vals:
            print(f"  {col}:")
            for v in dict.fromkeys(vals):  # dedupe, keep order
                print("    " + str(v)[:500].replace("\n", "\n    "))
    outs = _fmt_steps(r.get("function_outputs"))
    if outs:
        print("  tool output (last):")
        print("    " + str(outs[-1])[:600].replace("\n", "\n    "))

In [ ]:
# Diagnose failure modes + generate deterministic recommendations ---------------
import re
from datetime import datetime

def _nums(text):
    return set(re.findall(r"-?\d[\d,]*\.?\d*", str(text).replace(",", "")))

def diagnose(r):
    if r["evaluation_judgement"] == True:
        return None
    if r["status"] != "completed":
        return "agent_error"
    answer = str(r["actual_answer"]).lower()
    if any(k in answer for k in ("no data", "not found", "couldn't find", "unable to", "i cannot")):
        return "no_data_returned"
    expected_nums = _nums(r["expected_answer"])
    outputs = str(r.get("function_outputs") or "")
    if expected_nums and (expected_nums & _nums(outputs)):
        return "summarization_mismatch"   # right data retrieved, wrong final answer
    if r["evaluation_judgement"] == False:
        return "wrong_query"              # the generated query never surfaced the right data
    return "unclear_verdict"

RECOMMENDATIONS = {
    "summarization_mismatch": "Query returned the correct data but the final answer misstated it. Add to the agent instructions: 'Always state the exact figures returned by the query; never recalculate, round, or estimate.'",
    "wrong_query": "The generated query did not retrieve the expected data. For SQL/warehouse sources add an example query for this question pattern (validate with datasource.evaluate_few_shots). For all sources, add instructions mapping the business terms in the question to the exact tables/columns. Note: example queries are NOT supported for ontology data sources - use instructions instead.",
    "no_data_returned": "The agent found no data. Verify the data source includes the required tables, column types match the schema/ontology bindings (silent type mismatches can ingest zero rows), and the entities/relationships exist.",
    "agent_error": "The agent call itself failed. Verify the agent's workspace is on an ACTIVE, PAID F SKU capacity (trial SKUs are rejected) and the evaluating identity has access to the agent.",
    "unclear_verdict": "The judge could not decide. Make the expected answer specific and numeric, or tighten CRITIC_PROMPT with explicit acceptance criteria.",
    "high_latency": "Average answer time exceeds 60s. Reduce the number of tables/entities in the data source and apply semantic-model best practices (clear naming, described measures).",
}

detail["diagnosis"] = detail.apply(diagnose, axis=1)
diag_counts = detail["diagnosis"].dropna().value_counts()

recs = []
for diag, count in diag_counts.items():
    recs.append({"diagnosis": diag, "n_questions": int(count), "recommendation": RECOMMENDATIONS[diag]})
avg_latency = float(detail["execution_time_sec"].astype(float).mean() or 0)
if avg_latency > 60:
    recs.append({"diagnosis": "high_latency", "n_questions": int(len(detail)),
                 "recommendation": RECOMMENDATIONS["high_latency"]})

runs_meta = spark.read.table("aql_eval_runs").toPandas()
run_row = runs_meta[runs_meta["run_id"] == RUN_ID]
agent_name = str(run_row.iloc[0]["agent_name"]) if len(run_row) else "unknown"

if recs:
    print(f"=== RECOMMENDATIONS for {agent_name} (run {RUN_ID}) ===")
    for r in recs:
        print(f"\n[{r['diagnosis']}] ({r['n_questions']} question(s))")
        print("  " + r["recommendation"])
    spark.sql("""CREATE TABLE IF NOT EXISTS aql_recommendations (
        run_id STRING, agent_name STRING, diagnosis STRING, n_questions INT,
        recommendation STRING, created_at TIMESTAMP) USING DELTA""")
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
    rs = StructType([StructField("run_id", StringType()), StructField("agent_name", StringType()),
                     StructField("diagnosis", StringType()), StructField("n_questions", IntegerType()),
                     StructField("recommendation", StringType()), StructField("created_at", TimestampType())])
    now = datetime.utcnow()
    records = [(str(RUN_ID), agent_name, r["diagnosis"], r["n_questions"], r["recommendation"], now) for r in recs]
    spark.createDataFrame(records, schema=rs).write.mode("append").saveAsTable("aql_recommendations")
    print("\n(saved to aql_recommendations)")
else:
    print(f"All answers correct for {agent_name} - no recommendations.")

In [ ]:
# Config drift: did the agent's configuration change vs its previous run? -------
import difflib
import re as re2

runs_all = spark.read.table("aql_eval_runs").toPandas()
this_run = runs_all[runs_all["run_id"] == RUN_ID]
if this_run.empty:
    print("run metadata not found - skipping drift check")
else:
    a = this_run.iloc[0]
    prev = runs_all[(runs_all["agent_name"] == a["agent_name"]) &
                    (runs_all["workspace_name"] == a["workspace_name"]) &
                    (runs_all["status"] == "completed") &
                    (runs_all["started_at"] < a["started_at"])].sort_values("started_at")
    if prev.empty:
        print("no previous run for this agent - drift check skipped")
    else:
        prev_id = prev.iloc[-1]["run_id"]
        cfgs = (spark.read.table(OUTPUT_TABLE)
                .select("evaluation_id", "data_agent_etag", "data_agent_configuration")
                .filter(f"evaluation_id in ('{RUN_ID}', '{prev_id}')").toPandas())
        cur = cfgs[cfgs["evaluation_id"] == RUN_ID]
        prv = cfgs[cfgs["evaluation_id"] == prev_id]
        if cur.empty or prv.empty:
            print("config snapshots missing - drift check skipped")
        elif cur.iloc[0]["data_agent_etag"] == prv.iloc[0]["data_agent_etag"]:
            print("No configuration change vs previous run.")
        else:
            print("CONFIG DRIFT DETECTED: the agent's configuration changed since its previous evaluation.")
            def _instr(c):
                m = re2.match(r"DataAgentConfiguration\(instructions=(['""])(.*)\1, enable_preview_runtime=", str(c), re2.DOTALL)
                return m.group(2).replace("\\'", "'") if m else str(c)
            diff = difflib.unified_diff(
                _instr(prv.iloc[0]["data_agent_configuration"]).splitlines(),
                _instr(cur.iloc[0]["data_agent_configuration"]).splitlines(),
                fromfile="previous run", tofile="this run", lineterm="")
            for ln in list(diff)[:50]:
                print("  " + ln)
            print("\n-> If accuracy dropped after this change, consider rolling back to the previous instructions (shown above as '-' lines).")